# 02 · Structure Learning Deep Dive

**Owner: Member 2.**

Companion to §4 of `main.ipynb`. We test the *sensitivity* of
the learned DAG to hyperparameter choices — scoring function,
maximum in-degree, PC significance level — so we can defend
our final choice in the presentation.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
from src.data_loader import load_heart_disease
from src.preprocessing import (
    PreprocessConfig, build_dataset, train_test_split_df, variable_state_names,
)
from src.expert_network import build_expert_dag
from src.structure_learning import (
    StructureSearchConfig, learn_hill_climb, learn_pc,
    compare_structures, edge_set_diff,
)

df = build_dataset(load_heart_disease(), PreprocessConfig())
train, test = train_test_split_df(df, PreprocessConfig())
expert = build_expert_dag()

## Sensitivity to scoring function

In [ ]:
from src.visualization import plot_dag
dags = {}
for scoring in ['bic', 'k2', 'bdeu']:
    cfg = StructureSearchConfig(scoring=scoring)
    dags[f'HC-{scoring}'] = learn_hill_climb(train, cfg)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, g) in zip(axes, dags.items()):
    plot_dag(g, title=f'{name}  ({g.number_of_edges()} edges)', ax=ax)
plt.show()

## Sensitivity to maximum in-degree

Restricting in-degree is a common regularization technique —
too high and the model overfits, too low and it underfits.

In [ ]:
rows = []
for k in [1, 2, 3, 4, 5]:
    g = learn_hill_climb(train, StructureSearchConfig(max_indegree=k))
    rows.append({'max_indegree': k, 'edges': g.number_of_edges()})
pd.DataFrame(rows)

## Sensitivity to PC α

Higher α produces a denser skeleton.

In [ ]:
rows = []
for alpha in [0.001, 0.01, 0.05, 0.1]:
    g = learn_pc(train, StructureSearchConfig(pc_alpha=alpha))
    rows.append({'alpha': alpha, 'edges': g.number_of_edges()})
pd.DataFrame(rows)

## Whitelist / blacklist experiment

What if we *insist* that all `target → manifestation` expert
edges remain in the learned DAG?

In [ ]:
forced = [(u, v) for (u, v) in expert.edges() if u == 'target']
cfg = StructureSearchConfig(fixed_edges=forced)
constrained = learn_hill_climb(train, cfg)
print(f'Constrained DAG: {constrained.number_of_edges()} edges')

diff = edge_set_diff(expert, constrained)
print('Edges added by data on top of forced expert spine:')
for e in diff['only_in_learned']:
    print(' ', e[0], '↔', e[1])